# Model 8 — AdaBoost (AdaBoost.R2), Cases 1–3

**Idea:** build small trees one after another, each focusing on the rows the previous ones got wrong.

scikit-learn's `AdaBoostRegressor` uses **AdaBoost.R2**. In each round *t*:
1. Draw a training sample using the current row **weights** (all equal at the start) and fit a small tree.
2. Compute each row's **loss** Lᵢ = |error| ÷ largest |error| (`linear`), or its square (`square`), so 0 ≤ Lᵢ ≤ 1.
3. **Average loss** L̄ = Σ wᵢ Lᵢ. The tree must beat L̄ < 0.5, otherwise boosting stops.
4. **β** = L̄ / (1 − L̄); the tree's **vote weight** = learning_rate · ln(1/β). Lower loss → bigger vote.
5. **Re-weight:** wᵢ ← wᵢ · β^((1 − Lᵢ) · learning_rate), then normalise. Well-predicted rows shrink, hard rows (e.g. cloudy days) gain weight.

The final forecast is the **weighted median** of all trees' forecasts (not the mean).

**Tuned on validation (lowest RMSE):** base tree depth {3, 4, 5}, `learning_rate`, `loss` {linear, square}, and the number of rounds.

Same data, inputs, split, scoring window and metrics as the other notebooks.

In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostRegressor
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor

## 1. Settings

In [ ]:
# Works whether the notebook is opened from src/ or from the project folder.
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
CLEAN_CSV = ROOT / "Data" / "Clean" / "Winter_Clean.csv"
RESULTS = ROOT / "results"

CAPACITY_MW = 21.5
BAND_MW = 0.07 * CAPACITY_MW  # ±7% of capacity (GERC 2019 tolerance) = 1.505 MW

# Scoring window 07:00-18:15 (blocks 29-74): the plant never generates outside it.
DAY_FIRST_BLOCK, DAY_LAST_BLOCK = 29, 74

# Chronological split by target day (dates inclusive).
SPLITS = {
    "train": ("2025-10-15", "2026-01-14"),
    "val": ("2026-01-15", "2026-02-07"),
    "test": ("2026-02-08", "2026-02-14"),
}

# Short names for the CSV columns.
RENAME = {
    "Act. Power (MW)": "power",
    "W-GHI (W/m2)": "ghi",
    "W-MODT- (°C)": "modt",
    "W-AMBT (°C)": "ambt",
    "W-WSPD (m/s)": "wspd",
    "W-WDIR- (Degree)": "wdir",
}
WEATHER = ["ghi", "modt", "ambt", "wspd", "wdir"]

CASES = {
    "Case 1": ["power_D"],
    "Case 2": ["power_D"] + [f"{w}_D" for w in WEATHER],
    "Case 3": ["power_D"] + [f"{w}_D" for w in WEATHER] + [f"{w}_D+1" for w in WEATHER],
}

SEED = 42  # fixes the weighted resampling, so results are reproducible

# Candidate settings. Each model is fitted with MAX_ROUNDS; every smaller number of rounds is
# scored from the same fit with staged_predict (round t = the first t trees).
BASE_DEPTHS = [3, 4, 5]
LEARNING_RATES = [0.01, 0.05, 0.1, 0.5, 1.0]
LOSSES = ["linear", "square"]
MAX_ROUNDS = 400

## 2. Load the clean data and label each day with its split

In [ ]:
df = pd.read_csv(CLEAN_CSV, parse_dates=["TimeStamp", "Date"]).rename(columns=RENAME)

df["split"] = None
for name, (start, end) in SPLITS.items():
    df.loc[df["Date"].between(start, end), "split"] = name
assert df["split"].notna().all(), "some dates fall outside every split"

df.groupby("split")["Date"].agg(["min", "max", "nunique"]).loc[["train", "val", "test"]]

## 3. Build the inputs (identical to MLR)

`_D` = value at the same block on the previous day, `_D+1` = value on the target day (Case 3 only).

In [ ]:
by_block = df.groupby("Block")
for col in ["power"] + WEATHER:
    df[f"{col}_D"] = by_block[col].shift(1)
for w in WEATHER:
    df[f"{w}_D+1"] = df[w]

# 15 Oct has no previous day, so it has no inputs.
df = df.dropna(subset=["power_D"]).reset_index(drop=True)

daytime = df[df["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)].copy()
train = daytime[daytime["split"] == "train"]
val = daytime[daytime["split"] == "val"]

print("daytime rows per split:", daytime["split"].value_counts()[["train", "val", "test"]].to_dict())

## 4. Metrics

In [ ]:
def evaluate(actual, forecast):
    err = forecast - actual
    mae = err.abs().mean()
    rmse = np.sqrt((err**2).mean())
    # R² = 1 - SS_res / SS_tot: share of variation explained vs always predicting the mean.
    r2 = 1 - (err**2).sum() / ((actual - actual.mean()) ** 2).sum()
    return {
        "MAE (MW)": mae,
        "RMSE (MW)": rmse,
        "nMAE (%)": 100 * mae / CAPACITY_MW,
        "nRMSE (%)": 100 * rmse / CAPACITY_MW,
        "Within ±7% (%)": 100 * (err.abs() <= BAND_MW).mean(),
        "R² (%)": 100 * r2,
        "Blocks": len(err),
    }


def skill_score(rmse, rmse_persistence):
    # 0 = no better than persistence, positive = better, negative = worse.
    return 100 * (1 - rmse / rmse_persistence)


def rmse(actual, forecast):
    return np.sqrt(((np.clip(forecast, 0, CAPACITY_MW) - actual) ** 2).mean())


def ada_model(depth, learning_rate, loss, n_estimators):
    return AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=depth),
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        loss=loss,
        random_state=SEED,
    )

## 5. Tune on validation

For each case, every (base depth, learning_rate, loss) is fitted once with up to 400 rounds.
`staged_predict` then gives the forecast after round 1, 2, …, so every number of rounds is scored from that single fit.
The combination and number of rounds with the lowest validation RMSE are kept.

In [ ]:
search = []
for case, features in CASES.items():
    for depth, lr, loss in product(BASE_DEPTHS, LEARNING_RATES, LOSSES):
        model = ada_model(depth, lr, loss, MAX_ROUNDS).fit(train[features], train["power"])
        val_rmse = [rmse(val["power"], p) for p in model.staged_predict(val[features])]
        best_round = int(np.argmin(val_rmse)) + 1
        search.append({
            "case": case,
            "base_depth": depth,
            "learning_rate": lr,
            "loss": loss,
            "rounds fitted": len(model.estimators_),
            "best rounds": best_round,
            "val RMSE (MW)": val_rmse[best_round - 1],
        })
search = pd.DataFrame(search)

best = search.loc[search.groupby("case")["val RMSE (MW)"].idxmin()].set_index("case")
best

## 6. Fit the final model per case with its chosen settings

In [ ]:
models = {}
for case, features in CASES.items():
    b = best.loc[case]
    model = ada_model(int(b["base_depth"]), b["learning_rate"], b["loss"], int(b["best rounds"]))
    model.fit(train[features], train["power"])
    daytime[case] = np.clip(model.predict(daytime[features]), 0, CAPACITY_MW)
    models[case] = model

## 7. Round by round

For each case's final model: training and validation RMSE after every round, plus each round's average loss L̄
and vote weight. A round with L̄ close to 0.5 is barely better than guessing and gets a small vote.

In [ ]:
rounds = []
for case, features in CASES.items():
    m = models[case]
    train_curve = [rmse(train["power"], p) for p in m.staged_predict(train[features])]
    val_curve = [rmse(val["power"], p) for p in m.staged_predict(val[features])]
    for t in range(len(m.estimators_)):
        rounds.append({
            "case": case,
            "round": t + 1,
            "train RMSE (MW)": train_curve[t],
            "val RMSE (MW)": val_curve[t],
            "average loss": m.estimator_errors_[t],
            "vote weight": m.estimator_weights_[t],
        })
rounds = pd.DataFrame(rounds)

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex="col")
for i, (case, part) in enumerate(rounds.groupby("case")):
    axes[0, i].plot(part["round"], part["train RMSE (MW)"], label="Training")
    axes[0, i].plot(part["round"], part["val RMSE (MW)"], label="Validation")
    axes[0, i].set_title(case)
    axes[1, i].plot(part["round"], part["average loss"], color="tab:orange")
    axes[1, i].axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
    axes[1, i].set_xlabel("Boosting round")
axes[0, 0].set_ylabel("RMSE (MW)")
axes[1, 0].set_ylabel("Average loss L̄ (must be < 0.5)")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

## 8. Scores: every case on every split

In [ ]:
rows = []
for case in CASES:
    for split in ["train", "val", "test"]:
        part = daytime[daytime["split"] == split]
        scores = evaluate(part["power"], part[case])
        rmse_persistence = evaluate(part["power"], part["power_D"])["RMSE (MW)"]
        rows.append({
            "case": case,
            "split": split,
            **scores,
            "Skill vs persistence (%)": skill_score(scores["RMSE (MW)"], rmse_persistence),
        })

metrics = pd.DataFrame(rows).round(3)
metrics

Validation only, side by side (this is the table used to compare models):

In [ ]:
metrics[metrics["split"] == "val"].set_index("case").drop(columns=["split", "Blocks"])

## 9. Which inputs does the model rely on?

**Permutation importance on validation:** shuffle one input's values and measure how much validation RMSE rises.

In [ ]:
importance = {}
for case, features in CASES.items():
    result = permutation_importance(
        models[case], val[features], val["power"],
        scoring="neg_root_mean_squared_error", n_repeats=5, random_state=SEED,
    )
    importance[case] = pd.Series(result.importances_mean, index=features)
importance = pd.DataFrame(importance).reindex(CASES["Case 3"])
importance.round(3)  # rise in validation RMSE (MW) when the input is shuffled

## 10. Where does it fail? Worst validation days (Case 2, the realistic case)

In [ ]:
daily_mae = (
    daytime[daytime["split"] == "val"]
    .assign(abs_err=lambda x: (x["Case 2"] - x["power"]).abs())
    .groupby("Date")["abs_err"]
    .mean()
    .round(2)
)
daily_mae.sort_values(ascending=False).head(5)

In [ ]:
# Same three validation days as the other notebooks: clear, cloudy, clear.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, day in zip(axes, ["2026-01-25", "2026-01-26", "2026-01-27"]):
    d = daytime[daytime["Date"] == day]
    hours = (d["Block"] - 1) / 4
    ax.fill_between(hours, d["power"] - BAND_MW, d["power"] + BAND_MW, alpha=0.2, label="±7% band")
    ax.plot(hours, d["power"], color="black", label="Actual")
    ax.plot(hours, d["power_D"], ":", label="Persistence")
    for case in CASES:
        ax.plot(hours, d[case], "--", label=f"AdaBoost {case}")
    ax.set_title(day)
    ax.set_xlabel("Hour of day")
axes[0].set_ylabel("Power (MW)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Save forecasts, scores, the search and the per-round results

For the website's rounds slider: the forecast for every validation and test daytime block after selected rounds
(1, 2, 3, 5, 10, 20, 50, 100, 200 and the final round), in long format.

In [ ]:
(RESULTS / "predictions").mkdir(parents=True, exist_ok=True)
(RESULTS / "metrics").mkdir(parents=True, exist_ok=True)

# Every block, with night forecasts set to 0.
out = df[["TimeStamp", "Date", "Block", "split", "power"]].rename(columns={"power": "actual"})
for case in CASES:
    out[f"AdaBoost {case}"] = daytime[case].reindex(out.index).fillna(0.0)

SNAPSHOT_ROUNDS = [1, 2, 3, 5, 10, 20, 50, 100, 200]
holdout = daytime[daytime["split"].isin(["val", "test"])]
staged = []
for case, features in CASES.items():
    n = len(models[case].estimators_)
    keep = sorted({r for r in SNAPSHOT_ROUNDS if r <= n} | {n})
    for t, pred in enumerate(models[case].staged_predict(holdout[features]), start=1):
        if t in keep:
            staged.append(pd.DataFrame({
                "case": case, "round": t, "TimeStamp": holdout["TimeStamp"].values,
                "forecast": np.clip(pred, 0, CAPACITY_MW),
            }))
staged = pd.concat(staged, ignore_index=True)

out.to_csv(RESULTS / "predictions" / "adaboost.csv", index=False)
staged.to_csv(RESULTS / "predictions" / "adaboost_by_round.csv", index=False)
metrics.to_csv(RESULTS / "metrics" / "adaboost.csv", index=False)
search.to_csv(RESULTS / "metrics" / "adaboost_search.csv", index=False)
rounds.to_csv(RESULTS / "metrics" / "adaboost_rounds.csv", index=False)
importance.to_csv(RESULTS / "metrics" / "adaboost_importance.csv")